In [25]:
# Install PySpark
!pip install pyspark -q

# Celebal Technologies Summer Internship 2026
## Week 5: Apache Spark Fundamentals

**Objective:** Understand Spark fundamentals and perform data cleaning,
transformation, and aggregation using PySpark DataFrames on the Superstore dataset.

---

## Step 1: Why Spark over MapReduce?

**MapReduce Limitations:**
- Reads/writes data to disk after every operation — slow for iterative tasks
- High latency due to repeated disk I/O
- Complex programming model (verbose Java code)
- Not suitable for real-time/interactive analytics

**Spark Advantages:**
- **In-memory processing** — keeps data in RAM between operations, avoiding repeated disk reads
- Up to 100x faster than MapReduce for iterative algorithms (like ML)
- Easy-to-use DataFrame API (similar to SQL/Pandas)
- Supports batch, streaming, ML, and graph processing in one engine

In [26]:
!pip install pyspark -q

In [27]:
# ================================================
# Step 2: Initialize Spark Session
# ================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5_Superstore_Analysis") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")
print(f"Spark Version: {spark.version}")

Spark Session created successfully!
Spark Version: 4.0.2


In [28]:
# ================================================
# Step 3: Load Data into Spark DataFrame
# ================================================
from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.types import DoubleType

def load_data(filepath: str):
    """
    Loads a CSV file into a Spark DataFrame and ensures numeric
    columns (Sales, Quantity, Discount, Profit) are safely cast to
    numeric types, converting any malformed values to NULL instead
    of crashing the job.
    """
    df = spark.read.csv(
        filepath,
        header=True,
        inferSchema=True,
        multiLine=True,
        escape='"'
    )

    # Safely clean numeric columns - extract numeric pattern, then cast
    numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
    for c in numeric_cols:
        df = df.withColumn(
            c,
            regexp_extract(col(c).cast("string"), r'^-?\d*\.?\d+', 0).cast(DoubleType())
        )

    print(f"Data loaded successfully! Total rows: {df.count()}")
    return df

df = load_data("Sample - Superstore.csv")
df.show(5)
print("Columns:", df.columns)
df.printSchema()

Data loaded successfully! Total rows: 9994
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|

In [29]:
# ================================================
# Step 4: Data Cleaning Functions
# ================================================
from pyspark.sql.functions import sum as spark_sum, when

def remove_duplicates(df, subset_cols=None):
    """
    Removes duplicate rows from DataFrame.

    Args:
        df (DataFrame): Input Spark DataFrame
        subset_cols (list): Columns to check for duplicates.
                            If None, checks all columns.
    Returns:
        DataFrame: DataFrame without duplicates
    """
    before_count = df.count()
    df_clean = df.dropDuplicates(subset=subset_cols) if subset_cols else df.dropDuplicates()
    after_count = df_clean.count()
    print(f"Duplicates removed: {before_count - after_count} rows")
    return df_clean


def check_nulls(df):
    """Prints null value count for each column in DataFrame."""
    null_counts = df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
    null_counts.show()


def handle_nulls(df, strategy="drop", fill_value=None, columns=None):
    """
    Handles null values using drop or fill strategy.

    Args:
        df (DataFrame): Input Spark DataFrame
        strategy (str): 'drop' or 'fill'
        fill_value: Value to fill nulls with (used if strategy='fill')
        columns (list): Specific columns to apply strategy on
    Returns:
        DataFrame: Cleaned DataFrame
    """
    if strategy == "drop":
        return df.na.drop(subset=columns)
    elif strategy == "fill":
        return df.na.fill(fill_value, subset=columns)
    raise ValueError("Strategy must be 'drop' or 'fill'")


# Execute cleaning pipeline
print("=== Checking Null Values ===")
check_nulls(df)

print("\n=== Removing Duplicates ===")
df_clean = remove_duplicates(df)

print(f"\nFinal row count after cleaning: {df_clean.count()}")

=== Checking Null Values ===
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+-------

In [30]:
# ================================================
# Step 5: Filtering Functions
# ================================================

def filter_by_region(df, region: str):
    """Filters DataFrame by region."""
    return df.filter(df["Region"] == region)


def filter_by_category(df, category: str):
    """Filters DataFrame by product category."""
    return df.filter(df["Category"] == category)


def filter_by_discount_range(df, min_val: float, max_val: float):
    """Filters DataFrame by discount range (inclusive), ignoring nulls."""
    return df.filter(
        col("Discount").isNotNull() &
        (col("Discount") >= min_val) &
        (col("Discount") <= max_val)
    )


# Apply filters
print("=== Orders from West Region ===")
west_orders = filter_by_region(df_clean, "West")
west_orders.show(5)
print(f"Total West region orders: {west_orders.count()}")

print("\n=== Orders from Technology Category ===")
tech_orders = filter_by_category(df_clean, "Technology")
tech_orders.show(5)
print(f"Total Technology orders: {tech_orders.count()}")

print("\n=== Orders with Discount between 0.1 and 0.3 ===")
discount_orders = filter_by_discount_range(df_clean, 0.1, 0.3)
discount_orders.show(5)
print(f"Total filtered orders: {discount_orders.count()}")

=== Orders from West Region ===
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|   139|CA-2016-145583|10/13/2016|10/19/2016|Standard Class|   LC-16885|  Lena Creighton|   Consumer|United States|    Roseville|California|      95661|  West|OFF-SU-10001218|Office Suppli

In [31]:
# ================================================
# Step 7: Aggregation Functions
# ================================================
from pyspark.sql.functions import count, sum as spark_sum, avg, min as spark_min, max as spark_max

def get_summary_stats(df, column: str):
    """
    Calculates count, sum, avg, min, max for a given numeric column.

    Args:
        df (DataFrame): Input Spark DataFrame
        column (str): Column name to aggregate
    Returns:
        DataFrame: Summary statistics
    """
    return df.agg(
        count(column).alias(f"{column}_count"),
        spark_sum(column).alias(f"{column}_sum"),
        avg(column).alias(f"{column}_avg"),
        spark_min(column).alias(f"{column}_min"),
        spark_max(column).alias(f"{column}_max")
    )

print("=== Sales Summary Statistics ===")
sales_stats = get_summary_stats(df_clean, "Sales")
sales_stats.show()

print("=== Profit Summary Statistics ===")
profit_stats = get_summary_stats(df_clean, "Profit")
profit_stats.show()

=== Sales Summary Statistics ===
+-----------+------------------+------------------+---------+---------+
|Sales_count|         Sales_sum|         Sales_avg|Sales_min|Sales_max|
+-----------+------------------+------------------+---------+---------+
|       9994|2297200.8602999886|229.85800083049716|    0.444| 22638.48|
+-----------+------------------+------------------+---------+---------+

=== Profit Summary Statistics ===
+------------+-----------------+------------------+----------+----------+
|Profit_count|       Profit_sum|        Profit_avg|Profit_min|Profit_max|
+------------+-----------------+------------------+----------+----------+
|        9994|286397.0216999995|28.656896307784624| -6599.978|  8399.976|
+------------+-----------------+------------------+----------+----------+



In [32]:
# ================================================
# Step 8: GroupBy Operations
# ================================================

def group_and_aggregate(df, group_col: str, agg_col: str):
    """
    Groups DataFrame by a column and computes count, sum, avg of another column.

    Args:
        df (DataFrame): Input Spark DataFrame
        group_col (str): Column to group by
        agg_col (str): Column to aggregate
    Returns:
        DataFrame: Grouped and aggregated results
    """
    return df.groupBy(group_col).agg(
        count(agg_col).alias("count"),
        spark_sum(agg_col).alias("total_sum"),
        avg(agg_col).alias("average")
    ).orderBy("total_sum", ascending=False)


# Group by Category - total and avg Sales
print("=== Sales by Category ===")
category_sales = group_and_aggregate(df_clean, "Category", "Sales")
category_sales.show()

# Group by Region - total and avg Profit
print("=== Profit by Region ===")
region_profit = group_and_aggregate(df_clean, "Region", "Profit")
region_profit.show()

# Group by City - filter cities with count > 100 (Q6 requirement)
print("=== Cities with more than 100 orders ===")
city_counts = df_clean.groupBy("City").agg(count("*").alias("order_count"))
city_counts_filtered = city_counts.filter(col("order_count") > 100).orderBy("order_count", ascending=False)
city_counts_filtered.show()

=== Sales by Category ===
+---------------+-----+-----------------+------------------+
|       Category|count|        total_sum|           average|
+---------------+-----+-----------------+------------------+
|     Technology| 1847|836154.0329999991| 452.7092761234429|
|      Furniture| 2121|741999.7953000008| 349.8348869872705|
|Office Supplies| 6026|719047.0320000021|119.32410089611717|
+---------------+-----+-----------------+------------------+

=== Profit by Region ===
+-------+-----+------------------+------------------+
| Region|count|         total_sum|           average|
+-------+-----+------------------+------------------+
|   West| 3203|       108418.4489| 33.84903181392445|
|   East| 2848| 91522.78000000003| 32.13580758426967|
|  South| 1620|46749.430299999985|28.857673024691348|
|Central| 2323| 39706.36250000001|17.092708781747746|
+-------+-----+------------------+------------------+

=== Cities with more than 100 orders ===
+-------------+-----------+
|         City|orde

In [33]:
# ================================================
# Step 9: Schema Modification (Casting & Renaming)
# ================================================
from pyspark.sql.functions import to_timestamp

def rename_column(df, old_name: str, new_name: str):
    """Renames a column in the DataFrame."""
    return df.withColumnRenamed(old_name, new_name)


def cast_column(df, col_name: str, new_type: str):
    """Casts a column to a new data type."""
    return df.withColumn(col_name, col(col_name).cast(new_type))


# Demonstrate: Cast Order Date (string) to Date type and rename it
df_schema_demo = df_clean.withColumn(
    "order_date_parsed", to_timestamp(col("Order Date"), "M/d/yyyy")
)
df_schema_demo = rename_column(df_schema_demo, "order_date_parsed", "event_time")

print("=== Schema After Casting & Renaming ===")
df_schema_demo.select("Order ID", "Order Date", "event_time").show(5)
df_schema_demo.printSchema()

=== Schema After Casting & Renaming ===
+--------------+----------+-------------------+
|      Order ID|Order Date|         event_time|
+--------------+----------+-------------------+
|CA-2016-145583|10/13/2016|2016-10-13 00:00:00|
|US-2017-100930|  4/7/2017|2017-04-07 00:00:00|
|CA-2016-142545|10/28/2016|2016-10-28 00:00:00|
|CA-2015-130792| 4/28/2015|2015-04-28 00:00:00|
|CA-2016-166674|  4/1/2016|2016-04-01 00:00:00|
+--------------+----------+-------------------+
only showing top 5 rows
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: 

In [34]:
# ================================================
# Step 9 (continued): Handle Inconsistent Data
# ================================================

def remove_invalid_rows(df, null_check_col: str, empty_check_col: str):
    """
    Removes rows where null_check_col is NULL OR empty_check_col is an empty string.

    Args:
        df (DataFrame): Input Spark DataFrame
        null_check_col (str): Column to check for nulls
        empty_check_col (str): Column to check for empty strings
    Returns:
        DataFrame: DataFrame without invalid rows
    """
    return df.filter(
        col(null_check_col).isNotNull() & (col(empty_check_col) != "")
    )


# Demonstrate on Customer Name (null check) and City (empty check)
before = df_clean.count()
df_valid = remove_invalid_rows(df_clean, "Customer Name", "City")
after = df_valid.count()

print(f"Rows before validation: {before}")
print(f"Rows after validation: {after}")
print(f"Invalid rows removed: {before - after}")

Rows before validation: 9994
Rows after validation: 9994
Invalid rows removed: 0


In [35]:
# ================================================
# Step 10: Complete End-to-End Data Pipeline
# ================================================

def run_pipeline(filepath: str):
    """
    Complete data processing pipeline:
    1. Load data
    2. Clean data (remove duplicates, handle nulls)
    3. Filter data (West region only)
    4. Transform schema (cast Order Date to timestamp)
    5. Aggregate (total revenue by category)

    Args:
        filepath (str): Path to source CSV file
    Returns:
        DataFrame: Final aggregated result
    """
    print("Step 1: Loading data...")
    raw_df = load_data(filepath)

    print("\nStep 2: Cleaning data...")
    clean_df = remove_duplicates(raw_df)
    clean_df = handle_nulls(clean_df, strategy="fill", fill_value=0, columns=["Discount"])

    print("\nStep 3: Filtering data (West region)...")
    filtered_df = filter_by_region(clean_df, "West")

    print("\nStep 4: Transforming schema...")
    transformed_df = filtered_df.withColumn(
        "event_time", to_timestamp(col("Order Date"), "M/d/yyyy")
    )

    print("\nStep 5: Aggregating (Revenue by Category)...")
    result_df = transformed_df.groupBy("Category").agg(
        spark_sum("Sales").alias("total_revenue"),
        avg("Sales").alias("avg_sale"),
        count("*").alias("order_count")
    ).orderBy("total_revenue", ascending=False)

    return result_df


# Execute full pipeline
final_result = run_pipeline("Sample - Superstore.csv")
print("\n=== Final Pipeline Output: Revenue by Category (West Region) ===")
final_result.show()

Step 1: Loading data...
Data loaded successfully! Total rows: 9994

Step 2: Cleaning data...
Duplicates removed: 0 rows

Step 3: Filtering data (West region)...

Step 4: Transforming schema...

Step 5: Aggregating (Revenue by Category)...

=== Final Pipeline Output: Revenue by Category (West Region) ===
+---------------+------------------+------------------+-----------+
|       Category|     total_revenue|          avg_sale|order_count|
+---------------+------------------+------------------+-----------+
|      Furniture|252612.74350000036|357.30232461103304|        707|
|     Technology|251991.83199999982| 420.6875325542568|        599|
|Office Supplies|220853.24899999966| 116.4223769109118|       1897|
+---------------+------------------+------------------+-----------+



---
# Theory & Code Questions (Q1-Q15)
---

## Q1: Key limitations of MapReduce

**Answer:**
- **Disk I/O overhead:** MapReduce writes intermediate results to disk after every Map/Reduce stage, causing high latency
- **Not suited for iterative algorithms:** ML algorithms that need multiple passes over data become slow due to repeated disk reads
- **High latency:** Not designed for real-time or interactive queries
- **Complex programming model:** Requires verbose Java code for simple operations
- **No built-in support for streaming, graph, or ML workloads** — needs separate tools

Spark solves these by keeping data **in-memory** across operations, providing a unified engine for batch, streaming, ML, and graph processing.

## Q2: In-Memory Computing for Iterative ML Algorithms

**Answer:**
Traditional disk-based systems (like MapReduce) write data back to disk after every iteration. For ML algorithms (e.g., Gradient Descent, K-Means) that need **many iterations** over the same dataset, this causes massive disk I/O overhead.

Spark's **in-memory computing** keeps the dataset cached in RAM (using `.cache()` or `.persist()`) across iterations. Each iteration reads directly from memory instead of disk, making iterative ML algorithms **up to 100x faster**.

**Example:** In K-Means clustering, the same dataset is scanned repeatedly to recompute centroids. Spark caches this dataset once, and every iteration reuses the in-memory copy — avoiding repeated disk reads.

In [36]:
# ================================================
# Q3: Remove duplicates based on specific columns
# ================================================
# Using user_id and transaction_date as example columns
# (mapped to Customer ID and Order Date in our dataset)

df_q3 = df_clean.dropDuplicates(subset=["Customer ID", "Order Date"])
print(f"Rows after removing duplicates on Customer ID + Order Date: {df_q3.count()}")

Rows after removing duplicates on Customer ID + Order Date: 4992


In [37]:
# ================================================
# Q4: Filter region='West', group by category, find avg sale_amount
# ================================================

q4_result = df_clean.filter(col("Region") == "West") \
    .groupBy("Category") \
    .agg(avg("Sales").alias("avg_sale_amount")) \
    .orderBy("avg_sale_amount", ascending=False)

print("=== Average Sales by Category (West Region) ===")
q4_result.show()

=== Average Sales by Category (West Region) ===
+---------------+------------------+
|       Category|   avg_sale_amount|
+---------------+------------------+
|     Technology| 420.6875325542568|
|      Furniture|357.30232461103304|
|Office Supplies| 116.4223769109118|
+---------------+------------------+



## Q5: .na.drop() vs .na.fill()

**Answer:**
- **`.na.drop()`** — Removes rows that contain null values (in specified columns or any column)
- **`.na.fill()`** — Replaces null values with a specified default value, keeping the row

`.na.drop()` is used when missing data makes the row unusable. `.na.fill()` is used when we want to retain the row but substitute a safe default.

In [38]:
# Code example: Fill null values in 'status' column with 'Unknown'
# (Demonstrated using Ship Mode as example column)

df_q5 = df_clean.na.fill("Unknown", subset=["Ship Mode"])
df_q5.select("Ship Mode").show(5)

+--------------+
|     Ship Mode|
+--------------+
|Standard Class|
|Standard Class|
|Standard Class|
|Standard Class|
|  Second Class|
+--------------+
only showing top 5 rows


In [39]:
# ================================================
# Q6: Total count per city, only cities with count > 100
# ================================================

q6_result = df_clean.groupBy("City") \
    .agg(count("*").alias("total_count")) \
    .filter(col("total_count") > 100) \
    .orderBy("total_count", ascending=False)

print("=== Cities with more than 100 records ===")
q6_result.show()

=== Cities with more than 100 records ===
+-------------+-----------+
|         City|total_count|
+-------------+-----------+
|New York City|        915|
|  Los Angeles|        747|
| Philadelphia|        537|
|San Francisco|        510|
|      Seattle|        428|
|      Houston|        377|
|      Chicago|        314|
|     Columbus|        222|
|    San Diego|        170|
|  Springfield|        163|
|       Dallas|        157|
| Jacksonville|        125|
|      Detroit|        115|
+-------------+-----------+



## Q7: Immutability and Data Cleaning

**Answer:**
Spark DataFrames are **immutable** — once created, they cannot be modified in place. Operations like `.drop()`, `.withColumnRenamed()`, or `.filter()` do **not** change the original DataFrame; instead, they return a **new DataFrame** with the transformation applied.

**Impact on data cleaning:**
- Every cleaning step (dropping columns, renaming, filtering) must be **assigned to a new variable** or chained
- The original DataFrame remains unchanged, which is useful for debugging and reproducibility
- Encourages a **pipeline style** of programming: `df = df.filter(...).withColumn(...).drop(...)`

**Example:**
```python
df_dropped = df.drop("Postal Code")  # df is unchanged, df_dropped is new
```

In [40]:
# ================================================
# Q8: Filter age between 18-30 (inclusive) and subscription = 'Premium'
# ================================================
# Note: Superstore dataset doesn't have 'age' or 'subscription' columns,
# so we demonstrate the same filtering LOGIC using Quantity (numeric range)
# and Segment (categorical) as analogous columns.

q8_result = df_clean.filter(
    (col("Quantity") >= 2) & (col("Quantity") <= 5) &
    (col("Segment") == "Consumer")
)

print("=== Demonstration: Quantity between 2-5 AND Segment='Consumer' ===")
print("(Logic equivalent to: age between 18-30 AND subscription='Premium')")
q8_result.show(5)
print(f"Total matching rows: {q8_result.count()}")

=== Demonstration: Quantity between 2-5 AND Segment='Consumer' ===
(Logic equivalent to: age between 18-30 AND subscription='Premium')
+------+--------------+----------+----------+--------------+-----------+-----------------+--------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name| Segment|      Country|         City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name| Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+--------+-------------+-------------+----------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|   381|CA-2015-130792| 4/28/2015|  5/5/2015|Standard Class|   RA-19915|Russell Applegat

## Q9: Why handle nulls before aggregation?

**Answer:**
Handling null values **before** running aggregations like `sum()` or `avg()` is important because:

1. **Silent data loss:** Most aggregate functions (`sum`, `avg`) automatically skip nulls, which can silently skew results if nulls represent meaningful zeros or errors
2. **Misleading averages:** `avg()` divides by the count of **non-null** values, not total rows — so unhandled nulls can inflate the average artificially
3. **Downstream errors:** Nulls propagating into joins or further calculations can cause `NULL` results throughout the pipeline
4. **Business accuracy:** If nulls actually mean "0" (e.g., no discount applied), they should be filled before summing — otherwise the sum will be correct, but the row count/average will be wrong

**Best practice:** Always inspect and explicitly decide (drop vs fill) before aggregating, rather than relying on implicit null-skipping behavior.

In [41]:
# ================================================
# Q10: Cast raw_timestamp to TimestampType and rename to event_time
# ================================================
# Demonstrated using 'Order Date' as the raw_timestamp column

df_q10 = df_clean.withColumn(
    "raw_timestamp", to_timestamp(col("Order Date"), "M/d/yyyy")
).withColumnRenamed("raw_timestamp", "event_time")

print("=== Column casted and renamed: raw_timestamp -> event_time ===")
df_q10.select("Order Date", "event_time").printSchema()
df_q10.select("Order Date", "event_time").show(5)

=== Column casted and renamed: raw_timestamp -> event_time ===
root
 |-- Order Date: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+----------+-------------------+
|Order Date|         event_time|
+----------+-------------------+
|10/13/2016|2016-10-13 00:00:00|
|  4/7/2017|2017-04-07 00:00:00|
|10/28/2016|2016-10-28 00:00:00|
| 4/28/2015|2015-04-28 00:00:00|
|  4/1/2016|2016-04-01 00:00:00|
+----------+-------------------+
only showing top 5 rows


## Q11: Shuffle Process in Grouping Operations

**Answer:**
A **Shuffle** is the process of redistributing data **across partitions** (and across worker nodes in a cluster) so that rows with the same key end up on the same partition.

**During `groupBy()`:**
1. Data is initially scattered randomly across partitions
2. To group all rows with the same key (e.g., same `Category`), Spark must **physically move** matching rows to the same partition
3. This involves writing data to disk, transferring it over the network, and reading it back — which is expensive

**Why it's a "Wide Transformation":**
- A **narrow transformation** (like `filter()` or `map()`) processes each partition independently — no data movement needed
- A **wide transformation** (like `groupBy()`, `join()`, `distinct()`) requires data from **multiple partitions** to be combined, forcing a shuffle
- Shuffles are the **most expensive operation** in Spark due to network I/O, disk I/O, and serialization overhead

**Optimization tip:** Minimizing shuffles (e.g., using `reduceByKey` over `groupByKey`, or broadcasting small tables in joins) significantly improves performance.

In [42]:
# ================================================
# Q12: Remove rows where email is NULL OR username is empty string
# ================================================
# Demonstrated using Customer Name (proxy for email) and
# City (proxy for username) since dataset lacks email/username columns

df_q12 = df_clean.filter(
    ~(col("Customer Name").isNull() | (col("City") == ""))
)

print(f"Rows before: {df_clean.count()}")
print(f"Rows after removing null Customer Name OR empty City: {df_q12.count()}")

Rows before: 9994
Rows after removing null Customer Name OR empty City: 9994


In [43]:
# ================================================
# Q13: Use .agg() to calculate multiple statistics at once
# ================================================

q13_result = df_clean.agg(
    spark_min("Sales").alias("min_price"),
    spark_max("Sales").alias("max_price"),
    avg("Sales").alias("mean_price")
)

print("=== Multiple Statistics using .agg() on Sales column ===")
q13_result.show()

=== Multiple Statistics using .agg() on Sales column ===
+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|    0.444| 22638.48|229.85800083049716|
+---------+---------+------------------+



## Q14: Risk of inferSchema=true with messy date formats

**Answer:**
Using `inferSchema=true` tells Spark to **automatically guess** the data type of each column by sampling the data. This is risky with messy/inconsistent date formats because:

1. **Wrong type inference:** If a date column has mixed formats (e.g., `10/13/2016` and `2016-10-13`), Spark may infer it as `StringType` instead of `DateType`/`TimestampType`, since it cannot find one consistent pattern
2. **Silent failures:** Dates inferred as strings won't throw errors immediately, but later date-based operations (filtering, sorting by date) will fail or behave incorrectly
3. **Performance cost:** `inferSchema=true` requires Spark to **scan the entire dataset once** just to determine types, adding overhead before actual processing begins
4. **Inconsistent results across runs:** If schema inference samples different rows (in distributed reads), the inferred type might vary between executions
5. **Hidden data quality issues:** Malformed dates get silently treated as valid strings instead of being flagged as bad data

**Best practice:** In production pipelines, explicitly **define the schema** using `StructType` rather than relying on `inferSchema=true`, especially for date/timestamp columns. Parse dates explicitly using `to_timestamp()` with a known format string, as we did in Q10.

In [44]:
# ================================================
# Q15: Final Processing Pipeline
# 1. Filter duplicates
# 2. Fill null prices with 0
# 3. Group by store_id, calculate total revenue
# ================================================
# Demonstrated using Region as proxy for store_id, Sales as price

def final_pipeline(df):
    """
    Complete pipeline: dedupe -> fill nulls -> aggregate revenue.

    Args:
        df (DataFrame): Raw input DataFrame
    Returns:
        DataFrame: Total revenue per store (Region)
    """
    # Step 1: Remove duplicate rows
    df_step1 = df.dropDuplicates()

    # Step 2: Fill null Sales (price) values with 0
    df_step2 = df_step1.na.fill(0, subset=["Sales"])

    # Step 3: Group by Region (store_id) and calculate total revenue
    df_step3 = df_step2.groupBy("Region").agg(
        spark_sum("Sales").alias("total_revenue")
    ).orderBy("total_revenue", ascending=False)

    return df_step3


q15_result = final_pipeline(df_clean)
print("=== Final Pipeline: Total Revenue per Store (Region) ===")
q15_result.show()

=== Final Pipeline: Total Revenue per Store (Region) ===
+-------+------------------+
| Region|     total_revenue|
+-------+------------------+
|   West| 725457.8245000008|
|   East| 678781.2400000005|
|Central|501239.89079999976|
|  South| 391721.9050000001|
+-------+------------------+



---
# Summary & Key Insights
---

## What We Did
1. Loaded Superstore dataset (9994 rows) into a Spark DataFrame with robust numeric type handling
2. Performed data cleaning — checked nulls, removed duplicates
3. Applied filters on Region, Category, and Discount range
4. Computed aggregations (count, sum, avg, min, max) on Sales and Profit
5. Grouped data by Category, Region, and City with conditional filtering
6. Modified schema — cast Order Date to TimestampType and renamed to event_time
7. Validated data for nulls and empty values
8. Built a complete end-to-end pipeline: Load → Clean → Filter → Transform → Aggregate
9. Answered 15 theory + code questions covering Spark fundamentals

## Key Observations
- **Technology** category generates the highest average sale amount (₹452.7) despite fewer orders than Office Supplies
- **West region** is the top performer in both total Sales and Profit
- **Office Supplies** has the most orders (6026) but lowest average sale value — high volume, low margin
- 13 cities have more than 100 orders, with **New York City** leading (915 orders)
- The dataset required defensive parsing (regex-based numeric extraction) due to malformed CSV rows — a real-world data quality scenario
- No null/empty values were found in core columns (Customer Name, City), indicating a relatively clean source dataset

## Spark Concepts Demonstrated
| Concept | Where Used |
|---------|-----------|
| In-memory processing | Spark Session + cached DataFrame operations |
| Immutability | Every transformation returns a new DataFrame |
| Narrow transformations | `filter()`, `withColumn()` |
| Wide transformations (shuffle) | `groupBy()`, `dropDuplicates()` |
| Schema modification | `cast()`, `withColumnRenamed()` |
| Null handling | `.na.drop()`, `.na.fill()` |
| Aggregation | `.agg()`, `groupBy().agg()` |

## Real-World Data Quality Challenge Encountered
While loading the CSV, certain rows contained malformed values in the `Discount` column due to embedded commas in product names (CSV parsing edge case). This was resolved using `regexp_extract()` to safely extract numeric values, converting malformed entries to `NULL` instead of crashing the pipeline — a practical demonstration of **why null-handling before aggregation (Q9) and explicit schema control (Q14) matter** in production data engineering.